### Introduction
This notebook demonstrates how to use SRA public datasets hosted by AWS to get RNA-Seq gene counts file locations and their analysis objects attributes/QC metrics using Athena.   
This example uses boto3 and awswrangler python AWS clients instead of CLI tools.  
It shows possible metadata and analysis tables usage:
- Collecting gene counts from selecting run accessions by known run metadata attributes e.g. BioProject and assay type
- Flattering RNA-Seq feature count specific attributes and QC metrics

### AWS account and credentials
To run this tutorial you need:
- AWS account  
- role to use with appropriate permissions set, please see [this NCBI page](https://www.ncbi.nlm.nih.gov/sra/docs/sra-athena/) for more information  
- and default credentials and region set up, see e.g. [this AWS document](https://docs.aws.amazon.com/boto3/latest/guide/quickstart.html)
    - NOTE: please use us-east-1 region

#### Athena query output bucket - required
Must be in us-east-1

In [1]:
aws_output = 's3://<bucket-name-placeholder>/sra-athena-tutorial-output/'

### Source data location
**sra-pub-metadata-us-east-1** is AWS S3 public data bucket in **us-east-1** region containing gziped parquet files for  
- metadata table in sra/metadata subfolder
- analysis table in sra/analysis subfolder 

In [2]:
s3_bucket='sra-pub-metadata-us-east-1'

### Setup

In [3]:
%%capture
%env PIP_DISABLE_PIP_VERSION_CHECK=1

In [4]:
!pip3 install -q boto3[crt] awswrangler

In [5]:
import boto3
import botocore
import pandas as pd
import awswrangler as wr
import awswrangler.athena as wa
pd.set_option('display.max_colwidth', None)

### Specify Athena database name and output bucket
In case the role you are using does not have enough permissions to create new database please use existing one

In [6]:
aws_database='sra_athena_tutorial'
# output folder is required for Athena queries
# value below is a place holder used for test purposes, please provide your own
aws_output = 's3://sra-staging-metadata-us-east-1/sra-athena-tutorial-output/'

### Creating database if not exist

In [7]:
wr.catalog.create_database(name=aws_database, exist_ok=True, description="SRA Athena Tutorial")

### Creating *analysis* table if not exist

In [8]:
wr.catalog.create_parquet_table(
    database=aws_database,
    table="sra_analysis",
    path="s3://sra-pub-metadata-us-east-1/sra/analysis/",
    columns_types={
    	"acc": "STRING",
    	"study": "STRING",
    	"biosample": "STRING",
    	"bioproject": "STRING",
    	"alias": "STRING",
    	"center_name": "STRING",
    	"analysis_type": "STRING",
    	"target_run": "STRING",
    	"alignment": "STRUCT<includes_unaligned_reads:BOOLEAN, marks_duplicate_reads:BOOLEAN, includes_failed_reads:BOOLEAN>",
    	"standard_assembly": "STRING",
    	"pipe_section": "STRING",
    	"attributes": "ARRAY<STRUCT<k:STRING,v:STRING>>",
    	"files_processed": "BOOLEAN"
    },
    mode="overwrite",
    compression="gzip"
)

In [9]:
sql = "select count(*) c from sra_analysis"
df = wa.read_sql_query(sql=sql, database=aws_database, s3_output = aws_output)
print(f"{df.c[0]:,}")

4,577,181


### Creating *metadata* table if not exist

In [10]:
wr.catalog.create_parquet_table(
    database=aws_database,
    table="sra_metadata",
    path="s3://sra-pub-metadata-us-east-1/sra/metadata/",
    columns_types={
    	"acc": "STRING",
    	"assay_type": "STRING",
    	"center_name": "STRING",
    	"consent": "STRING",
    	"experiment": "STRING",
    	"sample_name": "STRING",
    	"instrument": "STRING",
    	"librarylayout": "STRING",
    	"libraryselection": "STRING",
    	"librarysource": "STRING",
    	"platform": "STRING",
    	"sample_acc": "STRING",
    	"biosample": "STRING",
    	"organism": "STRING",
    	"sra_study": "STRING",
    	"releasedate": "DATE",
    	"bioproject": "STRING",
    	"mbytes": "BIGINT",
    	"loaddate": "TIMESTAMP",
    	"avgspotlen": "BIGINT",
    	"mbases": "BIGINT",
    	"insertsize": "BIGINT",
    	"library_name": "STRING",
    	"biosamplemodel_sam": "ARRAY<STRING>",
    	"collection_date_sam": "DATE",
    	"geo_loc_name_country_calc": "STRING",
    	"geo_loc_name_country_continent_calc": "STRING",
    	"geo_loc_name_sam": "ARRAY<STRING>",
    	"ena_first_public_run": "ARRAY<STRING>",
    	"ena_last_update_run": "ARRAY<STRING>",
    	"sample_name_sam": "ARRAY<STRING>",
    	"datastore_filetype": "ARRAY<STRING>",
    	"datastore_provider": "ARRAY<STRING>",
    	"datastore_region": "ARRAY<STRING>",
    	"attributes": "ARRAY<STRUCT<k:STRING,v:STRING>>",
    	"run_file_version": "BIGINT",
    	"jattr": "STRING"
    },
    mode="overwrite",
    compression="gzip"
)

In [11]:
sql = "select count(*) c from sra_metadata"
df = wa.read_sql_query(sql=sql, database=aws_database, s3_output = aws_output)
print(f"{df.c[0]:,}")

44,654,914


### Quering for list of gene counts accessions and their locations by sra.metadata attributes

In [12]:
bioproject='PRJNA432002'

#### Either A: having feature counts specific attributes and QC metrics flattered/pivoted

In [13]:
sql=f"""
with X as (
SELECT 
acc, target_run, map_agg(item.k, case when item.k='alignment_rate' then split(item.v, '%')[1] else item.v end) as kv
FROM 
    sra_analysis cross join unnest(attributes) a(item) 
GROUP BY acc, target_run
), A as (
select 
    acc, target_run,
    kv['total_reads'] total_reads, 
    kv['layout'] layout,
    kv['intronic_reads'] intronic_reads,
    kv['intergenic_reads'] intergenic_reads,
    kv['mito_count'] mito_count,
    kv['decay_53_ratio'] decay_53_ratio,
    kv['bases'] bases,
    kv['bio_bases'] bio_bases,
    kv['strandness'] strandness,
    kv['alignment_rate'] alignment_rate,
    kv['unaligned_reads'] unaligned_reads,
    kv['discordant_pairs'] discordant_pairs
from X
),
Z as (  
  SELECT m.acc as run, 's3://sra-rnaseq-analysis/tsv/'||A.acc||'/'||m.acc||'.ncbi-gene-counts.tsv' path, a.*
  FROM sra_metadata m left join A on A.target_run = m.acc
  where m.bioproject = '{bioproject}' and (assay_type = 'RNA-Seq' or librarysource = 'TRANSCRIPTOMIC') and consent = 'public' and mbytes is not null
)
select * from Z
;
"""
dfa = wa.read_sql_query(sql=sql, database=aws_database, s3_output = aws_output)

##### Available analysis QC metrics and attributes

In [14]:
dfa.head(1).T

,0
run,SRR6605667
path,s3://sra-rnaseq-analysis/tsv/SRZ2017119/SRR6605667.ncbi-gene-counts.tsv
acc,SRZ2017119
target_run,SRR6605667
total_reads,405556
layout,PAIRED
intronic_reads,152620
intergenic_reads,45555
mito_count,18332
decay_53_ratio,0.0597


#### Note - it is possible not all runs will have gene counts available

In [15]:
print(dfa.path.isnull().sum())

9052


#### Or B: having feature counts specific attributes and QC metrics as json

In [16]:
sql = f"""
with A as (
    SELECT 
        acc, target_run, map_agg(item.k, case when item.k='alignment_rate' then split(item.v, '%')[1] else item.v end) as kv
    FROM 
        sra_analysis cross join unnest(attributes) a(item) 
    GROUP BY acc, target_run
)
SELECT m.acc as run, 's3://sra-rnaseq-analysis/tsv/'||A.acc||'/'||m.acc||'.ncbi-gene-counts.tsv' path, A.acc, json_format(cast(kv as JSON)) json
FROM sra_metadata m left join A on A.target_run = m.acc
where m.bioproject = '{bioproject}'
;
"""
dfb = wa.read_sql_query(sql=sql, database=aws_database, s3_output = aws_output)

#### Results
- run: SRA Run accession
- path: the path to gene counts file
- acc: SRA Analysis accession
- json: RNA-Seq feature counts QC metrics in json

In [17]:
df=dfb.head(1).T
df.style.set_properties(**{"text-align": "left"})

,0
run,SRR6547586
path,s3://sra-rnaseq-analysis/tsv/SRZ2322618/SRR6547586.ncbi-gene-counts.tsv
acc,SRZ2322618
json,"{""alignment_rate"":""89.18"",""bases"":""206052000"",""bio_bases"":""206052000"",""decay_53_ratio"":""0.0450"",""discordant_pairs"":""8434"",""intergenic_reads"":""36132"",""intronic_reads"":""113446"",""layout"":""PAIRED"",""mito_count"":""36726"",""strandness"":""none"",""total_reads"":""1030260"",""unaligned_reads"":""184922""}"
